<center><h1>Magdum_Seona_HW3</h1></center>
<br>
<br>

Name: Seona Magdum
<br> 
Github Username: seo2003
<br>
USC ID: 6077378254

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [24]:
import os

print(os.listdir("../data"))

['.DS_Store', 'bending1', 'walking', 'bending2', 'standing', 'sitting', 'lying', 'cycling']


Get the AReM Data Set

In [25]:
import os

for folder in os.listdir("../data"):
    if folder != ".DS_Store":
        print(folder, os.listdir("../data/" + folder)[:5])

bending1 ['dataset7.csv', 'dataset6.csv', 'dataset4.csv', 'dataset5.csv', 'dataset1.csv']
walking ['dataset7.csv', 'dataset6.csv', 'dataset4.csv', 'dataset5.csv', 'dataset1.csv']
bending2 ['dataset6.csv', 'dataset4.csv', 'dataset5.csv', 'dataset1.csv', 'dataset2.csv']
standing ['dataset7.csv', 'dataset6.csv', 'dataset4.csv', 'dataset5.csv', 'dataset1.csv']
sitting ['dataset7.csv', 'dataset6.csv', 'dataset4.csv', 'dataset5.csv', 'dataset1.csv']
lying ['dataset7.csv', 'dataset6.csv', 'dataset4.csv', 'dataset5.csv', 'dataset1.csv']
cycling ['dataset7.csv', 'dataset6.csv', 'dataset4.csv', 'dataset5.csv', 'dataset1.csv']


### (b) Test and Train Data

In [26]:
import os

#set relative path
data_path = "../data"

#training and testing file paths
train_files = []
test_files = []

#loop through each folder
for activity in sorted(os.listdir(data_path)):
    activity_path = os.path.join(data_path, activity)

    #skip if not folder
    if not os.path.isdir(activity_path):
        continue

    #loop through each file in folder
    for filename in sorted(os.listdir(activity_path)):

        #CSV files
        if not filename.endswith(".csv"):
            continue

        #relative path to current CSV file
        file_path = os.path.join(activity_path, filename)

        dataset_number = int(filename.replace("dataset", "").replace(".csv", ""))

        # For bending activities, use datasets 1 and 2 for testing
        if activity in ["bending1", "bending2"]:
            if dataset_number in [1, 2]:
                test_files.append(file_path)
            else:
                train_files.append(file_path)

        #use datasets 1, 2, and 3 for testing
        else:
            if dataset_number in [1, 2, 3]:
                test_files.append(file_path)
            else:
                train_files.append(file_path)

print("Training files:", len(train_files))
print("Testing files:", len(test_files))

Training files: 69
Testing files: 19


### (c) Feature Extraction

#### i. Research

Time_domain features are used to summarize the data's distribution over the time variable. Commonly used features include the minimum, which is the smallest value in the series, and the maximum, which is the largest. There are also the mean, median, standard deviation, variance, range, and the IQR. Other than the basic statistical features, there are some energy features, including the root mean square (RMS), which measures the magnitude of the signal, and the energy, which measures the total squared magnitude of the signal. 

#### ii. Extraction

In [27]:
import pandas as pd

file_path = "../data/bending1/dataset1.csv"

df = pd.read_csv(file_path, skiprows=4)

print(df.head())
print(df.columns)
print(df.shape)

   # Columns: time  avg_rss12  var_rss12  avg_rss13  var_rss13  avg_rss23  \
0                0      39.25       0.43      22.75       0.43      33.75   
1              250      39.25       0.43      23.00       0.00      33.00   
2              500      39.25       0.43      23.25       0.43      33.00   
3              750      39.50       0.50      23.00       0.71      33.00   
4             1000      39.50       0.50      24.00       0.00      33.00   

   var_rss23  
0        1.3  
1        0.0  
2        0.0  
3        0.0  
4        0.0  
Index(['# Columns: time', 'avg_rss12', 'var_rss12', 'avg_rss13', 'var_rss13',
       'avg_rss23', 'var_rss23'],
      dtype='object')
(480, 7)


In [28]:
import os
import pandas as pd

data_path = "../data"

#six sensor measurements
sensor_columns = [
    "avg_rss12", "var_rss12",
    "avg_rss13", "var_rss13",
    "avg_rss23", "var_rss23"
]

#empty list to store the features for each instance
all_features = []

for activity in sorted(os.listdir(data_path)):
    activity_path = os.path.join(data_path, activity)

    if not os.path.isdir(activity_path):
        continue

    for filename in sorted(os.listdir(activity_path)):

        #only CSV files
        if not filename.endswith(".csv"):
            continue

        #create the relative path to the current CSV file
        file_path = os.path.join(activity_path, filename)

        #bending2 dataset4 file, which used spaces instead of commas
        if activity == "bending2" and filename == "dataset4.csv":
            df = pd.read_csv(file_path, skiprows=5, sep=r"\s+", header=None)

            # Assign column names
            df.columns = [
                "time", "avg_rss12", "var_rss12",
                "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"
            ]

        else:
            #load the CSV file and skip the four metadata rows and use only the first seven columns
            df = pd.read_csv(file_path, skiprows=4, usecols=range(7))

        #store the instance name and its corresponding activity label
        row = {
            "instance": activity + "_" + filename.replace(".csv", ""),
            "activity": activity
        }

        #calculate statistical features
        for i, column in enumerate(sensor_columns, start=1):
            values = df[column]
            row[f"min_{i}"] = values.min()
            row[f"max_{i}"] = values.max()
            row[f"mean_{i}"] = values.mean()
            row[f"median_{i}"] = values.median()
            row[f"std_{i}"] = values.std()
            row[f"q1_{i}"] = values.quantile(0.25)
            row[f"q3_{i}"] = values.quantile(0.75)
        all_features.append(row)

features_df = pd.DataFrame(all_features)

print("Dataset shape:", features_df.shape)
features_df.head()

Dataset shape: (88, 44)


,instance,activity,min_1,max_1,mean_1,median_1,std_1,q1_1,q3_1,min_2,...,std_5,q1_5,q3_5,min_6,max_6,mean_6,median_6,std_6,q1_6,q3_6
0,bending1_dataset1,bending1,37.25,45.00,40.624792,40.50,1.476967,39.25,42.00,0.0,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.43,0.582915,0.0,1.30
1,bending1_dataset2,bending1,38.00,45.67,42.812812,42.50,1.435550,42.00,43.67,0.0,...,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.43,0.601010,0.0,1.30
2,bending1_dataset3,bending1,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,0.0,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.0,0.94
3,bending1_dataset4,bending1,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,0.0,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.0,1.00
4,bending1_dataset5,bending1,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,0.0,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.0,0.50


In [29]:
print("Dataset shape:", features_df.shape)
print("Instances:", len(features_df))
print("Feature columns:", len(features_df.columns) - 2)
print("Missing values:", features_df.isnull().sum().sum())

Dataset shape: (88, 44)
Instances: 88
Feature columns: 42
Missing values: 0


In [30]:
features_df.to_csv("time_domain_features.csv", index=False)

#### iii. Standard Deviation

In [31]:
import numpy as np
import pandas as pd

#select 42 feature columns
feature_columns = [
    col for col in features_df.columns
    if col not in ["instance", "activity"]
]

#sd of all 88 instances
standard_deviations = features_df[feature_columns].std()

#bootstrap 
n_bootstrap = 10000
np.random.seed(42)

bootstrap_results = []

for column in feature_columns:
    values = features_df[column].values
    n = len(values)

    bootstrap_stds = []

    for i in range(n_bootstrap):
        sample = np.random.choice(values, size=n, replace=True)

        #sd of bootstrap sample
        bootstrap_stds.append(np.std(sample, ddof=1))

    #90% CI
    lower = np.percentile(bootstrap_stds, 5)
    upper = np.percentile(bootstrap_stds, 95)

    bootstrap_results.append({
        "Feature": column,
        "Standard Deviation": standard_deviations[column],
        "90% CI Lower": lower,
        "90% CI Upper": upper
    })

results_df = pd.DataFrame(bootstrap_results)

print(results_df.round(4).to_string(index=False))

 Feature  Standard Deviation  90% CI Lower  90% CI Upper
   min_1              9.5700        8.2646       10.7562
   max_1              4.3944        3.3215        5.2776
  mean_1              5.3357        4.7094        5.8841
median_1              5.4401        4.7815        5.9950
   std_1              1.7722        1.5703        1.9553
    q1_1              6.1536        5.5612        6.6371
    q3_1              5.1389        4.3368        5.8511
   min_2              0.0000        0.0000        0.0000
   max_2              5.0627        4.6195        5.3998
  mean_2              1.5742        1.3921        1.7066
median_2              1.4122        1.2347        1.5444
   std_2              0.8841        0.8033        0.9415
    q1_2              0.9464        0.8304        1.0369
    q3_2              2.1253        1.8938        2.2985
   min_3              2.9565        2.7610        3.1095
   max_3              4.8751        4.1713        5.4763
  mean_3              4.0084   

#### iv. Select Features

Based on the extracted time domain features, I selected minimum, mean, and maximum as the three most important feature types. The minimum captures the lowest observed value, the maximum captures the peak value, and the mean captures the average sensor measurement across the instances. These features would be best for describing the range of sensor activity with different movement patterns. The bootstrap results also show that variability differs across features: for example, min_1 has a standard deviation of 9.57, where min_2 and min_4 have standard deviations of zero. 

## 2. ISLR 3.7.4

### (a) Linear Train

We would expect the cubic regression's training RSS to be lower than the linear regression's training RSS. The cubic model includes the same terms as the linear model with an additional x^2 and x^3, meaning it can fit into a wider variety of curves. Since a smaller RSS means it would fit the training data better, the cubic regression should have a lower than or equal training RSS compared to the linear regression. This is because the cubic model is more flexible and can fit the data at least as well as the linear model.

### (b) Linear Test

When using a test RSS rather than a training RSS, the linear model would have a lower RSS since the true relationship is linear. The cubic model could lead to overfitting and overall worse performance, leading to a higher RSS than the linear. 

### (c) Not Linear Train

If we dont know the relationship between X and Y, the cubic regression training RSS would be less than the linear training. Regardless of knowing if the true relationship is linear or nonlinear, the cubic model includes the linear model, meaning it can fit the training data at least as well as the linear model, so its training RSS cannot be higher. 

### (d) Not Linear Testing

If the true relationship is nonlinear, we would not have enough information to tell which model would have the lower RSS. If the true relationship is closer to a cubic model, the cubic model will have a lower RSS. But if the true relationship leans closer to linear, the linear model will have a lower RSS. 